# 🔬 Notebook 5a: Ablation 1 — Remove Soft Mask KL Loss (w/o $L_{\text{KL}}$)
This notebook runs **Ablation 1**: Knowledge Distillation on Crack500 with **Mask KL Loss disabled** ($lpha = 0$).
* **Goal**: Measure how much performance and OOD generalization drop when soft KL logit distillation is removed.
* **Input Dataset**: Crack500 + pre-computed SAM 2 teacher logits/features.


In [ ]:
!mkdir -p configs utils distillation scripts checkpoints data/datasets data/teacher_logits_box runs


In [ ]:
!pip install -q ultralytics albumentations pycocotools thop pyyaml pandas


In [ ]:
# Auto-fetch project modules to ensure 100% self-contained execution on Kaggle
!git clone https://github.com/shahin1717/crackdistill.git repo_code || true
!cp -r repo_code/distillation repo_code/utils repo_code/configs repo_code/scripts .


In [ ]:
import os, shutil
from pathlib import Path
input_dir = Path("/kaggle/input/distill_datasetforme")
if not input_dir.exists(): input_dir = Path("/kaggle/input")
datasets_dir = Path("data/datasets")
datasets_dir.mkdir(parents=True, exist_ok=True)

for root, dirs, files in os.walk(str(input_dir)):
    root_path = Path(root)
    if "traincrop" in dirs:
        dest = datasets_dir / "crack500"
        if os.path.lexists(dest): os.unlink(dest) if os.path.islink(dest) else shutil.rmtree(dest)
        os.symlink(root_path, dest)
        print(f"Linked Crack500: {root_path} -> {dest}")
        break


In [ ]:
# Run Ablation 1 (No Mask KL)
import sys
sys.path.insert(0, ".")
from distillation.kd_trainer import KDSegmentationTrainer
from utils.config_loader import load_config, override_config

cfg = load_config("configs/config.yaml")
cfg = override_config(cfg, {
    "project.name": "crack_distill",
    "project.experiment": "ablation_no_mask_kd",
    "distillation.enabled": True,
    "distillation.losses.mask_kd.enabled": False,
    "distillation.losses.feature.enabled": True,
    "distillation.losses.boundary.enabled": True,
    "teacher.logits_dir": "data/teacher_logits_box/"
})

trainer = KDSegmentationTrainer(cfg)
trainer.train()
print("✓ Ablation 1 (No Mask KL) completed!")
